In [1]:
import gc
import random
import h5py
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, CLIPModel, DepthAnythingForDepthEstimation, CLIPVisionModel
from tqdm.notebook import tqdm
import os
import shutil

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")
torch.backends.cudnn.benchmark = True

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE   = 224
BATCH_SIZE = 8
EPOCHS     = 50
LR         = 1e-4
DEPTH_MIN  = 1e-3
DATA_PATH  = "/kaggle/input/datasets/rmzhang0526/nyu-depth-v2-labeled/nyu_depth_v2_labeled.mat"

# "A" = final layer only  (simple single-layer decoder)
# "B" = DPT evenly spaced (proper DPT decoder, layers 3,6,9,12)
# "C" = Phase 1 informed  (proper DPT decoder, our selected layers)

ACTIVE_CONDITION = "C"

LAYER_CONFIGS = {
    "dinov2": {
        "A": [12],
        "B": [3, 6, 9, 12],
        "C": [2, 4, 8, 12],
    },
    "clip": {
        "A": [12],
        "B": [3, 6, 9, 12],
        "C": [2, 4, 8, 11],
    },
    "depth_anything": {
        "A": [12],
        "B": [3, 6, 9, 12],
        "C": [2, 5, 9, 12],
    },
    "vit_small": {
        "A": [12],
        "B": [3, 6, 9, 12],
        "C": [2, 4, 7, 10],
    },
}

print(f"Device: {DEVICE}")
print(f"Active condition: {ACTIVE_CONDITION}")
for m, cfg in LAYER_CONFIGS.items():
    print(f"  {m:20s} | layers = {cfg[ACTIVE_CONDITION]}")

Device: cuda
Active condition: C
  dinov2               | layers = [2, 4, 8, 12]
  clip                 | layers = [2, 4, 8, 11]
  depth_anything       | layers = [2, 5, 9, 12]
  vit_small            | layers = [2, 4, 7, 10]


In [2]:
class NYUDataset(Dataset):
    def __init__(self, records):
        self.records   = records
        self.transform = T.Compose([
            T.ToPILImage(),
            T.Resize((IMG_SIZE, IMG_SIZE)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406],
                        std =[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec   = self.records[idx]
        image = self.transform(rec["image"])
        depth = torch.from_numpy(rec["depth"]).unsqueeze(0)
        depth = F.interpolate(
            depth.unsqueeze(0),
            size=(IMG_SIZE, IMG_SIZE),
            mode="bilinear", align_corners=False
        ).squeeze(0)
        return {"image": image, "depth": depth}


def load_nyu(path, seed=SEED):
    print("Loading NYUv2...")
    with h5py.File(path, "r") as f:
        images = np.moveaxis(np.array(f["images"]), 1, -1).astype(np.uint8)
        depths = np.array(f["depths"]).astype(np.float32)
    rng = np.random.default_rng(seed)
    idx = np.arange(len(images))
    rng.shuffle(idx)
    n_val       = int(len(idx) * 0.2)
    train_idx   = idx[n_val:]
    val_idx     = idx[:n_val]
    train_recs  = [{"image": images[i], "depth": depths[i]} for i in train_idx]
    val_recs    = [{"image": images[i], "depth": depths[i]} for i in val_idx]
    print(f"Train: {len(train_recs)}  Val: {len(val_recs)}")
    return train_recs, val_recs


train_records, val_records = load_nyu(DATA_PATH)
train_dataset = NYUDataset(train_records)
val_dataset   = NYUDataset(val_records)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                           shuffle=True,  num_workers=2, pin_memory=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                           shuffle=False, num_workers=2, pin_memory=True)

Loading NYUv2...
Train: 1160  Val: 289


In [3]:
def load_backbone(model_name):
    if model_name == "dinov2":
        model = AutoModel.from_pretrained(
            "facebook/dinov2-base", output_hidden_states=True)
        meta = {"patch_size": 14, "num_layers": 12, "hidden_dim": 768}

    elif model_name == "clip":
        model = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch16")
        meta  = {"patch_size": 16, "num_layers": 12, "hidden_dim": 768}

    elif model_name == "depth_anything":
        full  = DepthAnythingForDepthEstimation.from_pretrained(
            "LiheYoung/depth-anything-base-hf")
        model = full.backbone
        model.config.output_hidden_states = True
        meta  = {"patch_size": 14, "num_layers": 12, "hidden_dim": 768}

    elif model_name == "vit_small":
        model = AutoModel.from_pretrained(
            "WinKawaks/vit-small-patch16-224", output_hidden_states=True)
        meta  = {"patch_size": 16, "num_layers": 12, "hidden_dim": 384}

    else:
        raise ValueError(f"Unknown model: {model_name}")

    meta["n_patches"] = IMG_SIZE // meta["patch_size"]
    model = model.to(DEVICE).eval()
    for p in model.parameters():
        p.requires_grad_(False)

    print(f"Loaded {model_name} | "
          f"dim={meta['hidden_dim']}  "
          f"patch={meta['patch_size']}  "
          f"n_patches={meta['n_patches']}")
    return model, meta

DPT Decoder

In [4]:
# Component 1: Readout projection (CLS → patches)
class ReadoutProject(nn.Module):
    """
    DPT 'project' readout: concatenate CLS token with each patch token,
    then project back to original dim.
    """
    def __init__(self, dim):
        super().__init__()
        self.norm = nn.LayerNorm(dim)        
        self.proj = nn.Sequential(
            nn.Linear(2 * dim, dim),
            nn.GELU(),
        )

    def forward(self, tokens):
        # tokens: (B, 1+N, D) — position 0 is CLS
        tokens  = self.norm(tokens)           
        cls     = tokens[:, 0:1, :].expand(-1, tokens.shape[1] - 1, -1)
        patches = tokens[:, 1:, :]
        return self.proj(torch.cat([patches, cls], dim=-1))  # (B, N, D)


# Component 2: Reassemble block
class ReassembleBlock(nn.Module):
    """
    Tokens → spatial feature map at target resolution.
    factor=4  → ConvTranspose upsample 4x  (shallowest layer)
    factor=2  → ConvTranspose upsample 2x
    factor=1  → keep same resolution
    factor=0.5 → Conv stride-2 downsample  (deepest layer)
    """
    def __init__(self, in_dim, out_dim, n_patches, factor):
        super().__init__()
        self.n_patches = n_patches
        self.readout   = ReadoutProject(in_dim)
        self.proj      = nn.Conv2d(in_dim, out_dim, kernel_size=1)

        if factor == 4:
            self.resample = nn.ConvTranspose2d(out_dim, out_dim, kernel_size=4, stride=4)
        elif factor == 2:
            self.resample = nn.ConvTranspose2d(out_dim, out_dim, kernel_size=2, stride=2)
        elif factor == 1:
            self.resample = nn.Identity()
        elif factor == 0.5:
            self.resample = nn.Conv2d(out_dim, out_dim, kernel_size=3, stride=2, padding=1)
        else:
            raise ValueError(f"Unsupported factor: {factor}")

    def forward(self, tokens):
        x = self.readout(tokens)                              # (B, N, D)
        B, N, D = x.shape
        x = x.permute(0, 2, 1).reshape(B, D, self.n_patches, self.n_patches)
        x = self.proj(x)
        x = self.resample(x)
        return x


# Component 3: ResidualConvUnit
class ResidualConvUnit(nn.Module):
    """Two conv3x3 with residual skip — the RefineNet building block."""
    def __init__(self, features):
        super().__init__()
        self.conv1 = nn.Conv2d(features, features, 3, padding=1, bias=False)
        self.conv2 = nn.Conv2d(features, features, 3, padding=1, bias=False)
        self.act   = nn.GELU()

    def forward(self, x):
        return x + self.conv2(self.act(self.conv1(self.act(x))))


# Component 4: FusionBlock
class FusionBlock(nn.Module):
    def __init__(self, features):
        super().__init__()
        self.rcu_deep  = ResidualConvUnit(features)   
        self.rcu_fused = ResidualConvUnit(features)   

    def forward(self, skip, x=None):
        out = skip                                   
        if x is not None:
            out = out + self.rcu_deep(x)             
        out = self.rcu_fused(out)
        out = F.interpolate(out, scale_factor=2,
                            mode="bilinear", align_corners=True)
        return out


# Full DPT decoder (Conditions B and C)
class DPTDecoder(nn.Module):
    """
    Proper DPT decoder.
    layer_indices: list of 4 ints (1-based), ordered shallowest → deepest.
    Reassemble factors are always [4, 2, 1, 0.5] for the 4 selected layers.
    """
    FUSION_DIM = 256
    FACTORS    = [4, 2, 1, 0.5] 

    def __init__(self, hidden_dim, n_patches, layer_indices):
        super().__init__()
        assert len(layer_indices) == 4
        self.layer_indices = layer_indices
        F_ = self.FUSION_DIM

        self.reassemble = nn.ModuleList([
            ReassembleBlock(hidden_dim, F_, n_patches, factor)
            for factor in self.FACTORS
        ])
        self.fusion = nn.ModuleList([FusionBlock(F_) for _ in range(4)])

        self.head = nn.Sequential(
            nn.Conv2d(F_, F_ // 2, kernel_size=3, padding=1),
            nn.Upsample(size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=True),
            nn.Conv2d(F_ // 2, 32, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(32, 1, kernel_size=1),
        )

    def forward(self, hidden_states):
        features = [
            self.reassemble[i](hidden_states[layer_idx])
            for i, layer_idx in enumerate(self.layer_indices)
        ]
        x = self.fusion[3](features[3])          
        x = self.fusion[2](features[2], x)
        x = self.fusion[1](features[1], x)
        x = self.fusion[0](features[0], x)
        return F.relu(self.head(x)) + DEPTH_MIN


# Simple single-layer decoder (Condition A)
class SingleLayerDecoder(nn.Module):
    FUSION_DIM = 256

    def __init__(self, hidden_dim, n_patches, layer_index):
        super().__init__()
        self.layer_index = layer_index
        F_ = self.FUSION_DIM
        self.readout = ReadoutProject(hidden_dim)
        self.proj    = nn.Conv2d(hidden_dim, F_, kernel_size=1)
        self.up1     = nn.Sequential(ResidualConvUnit(F_), nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True))
        self.up2     = nn.Sequential(ResidualConvUnit(F_), nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True))
        self.up3     = nn.Sequential(ResidualConvUnit(F_), nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True))
        self.up4     = nn.Sequential(ResidualConvUnit(F_), nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True))
        self.head    = nn.Sequential(
            nn.Conv2d(F_, F_ // 2, 3, padding=1),
            nn.Upsample(size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=True),
            nn.Conv2d(F_ // 2, 32, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(32, 1, 1),
        )
        self.n_patches = n_patches

    def forward(self, hidden_states):
        tokens = hidden_states[self.layer_index]
        x = self.readout(tokens)
        B, N, D = x.shape
        x = x.permute(0, 2, 1).reshape(B, D, self.n_patches, self.n_patches)
        x = self.proj(x)
        x = self.up1(x); x = self.up2(x); x = self.up3(x); x = self.up4(x)
        return F.relu(self.head(x)) + DEPTH_MIN

In [5]:
def silog_loss(pred, target, variance_focus=0.85):
    if pred.shape != target.shape:
        target = F.interpolate(target, size=pred.shape[2:],
                               mode="bilinear", align_corners=False)
    valid = (target > DEPTH_MIN) & (pred > DEPTH_MIN)
    if valid.sum() == 0:
        return (pred * 0).mean()
    d = torch.log(pred[valid]) - torch.log(target[valid])
    return torch.sqrt((d ** 2).mean() - variance_focus * (d.mean() ** 2))


@torch.no_grad()
def compute_metrics(pred, target):
    if pred.shape != target.shape:
        target = F.interpolate(target, size=pred.shape[2:],
                               mode="bilinear", align_corners=False)
    valid = (target > DEPTH_MIN) & (pred > DEPTH_MIN)
    if valid.sum() == 0:
        return {"abs_rel": float("nan"),
                "rmse":    float("nan"),
                "delta1":  float("nan")}
    p = pred[valid]
    t = target[valid]
    abs_rel = float((torch.abs(p - t) / t).mean())
    rmse    = float(torch.sqrt(((p - t) ** 2).mean()))
    delta1  = float(((torch.max(p / t, t / p)) < 1.25).float().mean())
    return {"abs_rel": abs_rel, "rmse": rmse, "delta1": delta1}

In [6]:
def build_decoder(condition, model_name, meta):
    layer_indices = LAYER_CONFIGS[model_name][condition]
    if condition == "A":
        decoder = SingleLayerDecoder(
            hidden_dim   = meta["hidden_dim"],
            n_patches    = meta["n_patches"],
            layer_index  = layer_indices[0],
        )
    else:
        decoder = DPTDecoder(
            hidden_dim    = meta["hidden_dim"],
            n_patches     = meta["n_patches"],
            layer_indices = layer_indices,
        )
    n_params = sum(p.numel() for p in decoder.parameters())
    print(f"  Decoder params: {n_params/1e6:.2f}M")
    return decoder.to(DEVICE)

In [7]:
@torch.no_grad()
def get_clip_hidden_states(model, images):
    if images.ndim == 3:
        images = images.unsqueeze(0)
    outputs = model(pixel_values=images, output_hidden_states=True)
    return outputs.hidden_states      # identical interface to all other models

In [8]:
def run_experiment(model_name, condition, epochs=EPOCHS):
    print(f"  {model_name.upper()} | Condition {condition} "
          f"| layers={LAYER_CONFIGS[model_name][condition]}")

    backbone, meta = load_backbone(model_name)
    decoder        = build_decoder(condition, model_name, meta)
    layer_indices  = LAYER_CONFIGS[model_name][condition]

    optimizer = torch.optim.AdamW(
        decoder.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr           = LR,
        steps_per_epoch  = len(train_loader),
        epochs           = epochs,
    )

    best_abs_rel = float("inf")
    best_state   = None
    history      = []

    for epoch in range(epochs):
        decoder.train()
        train_loss = 0.0

        for batch in tqdm(train_loader,
                          desc=f"Epoch {epoch+1}/{epochs}", leave=False):
            images   = batch["image"].to(DEVICE, non_blocking=True)
            depth_gt = batch["depth"].to(DEVICE, non_blocking=True)

            with torch.no_grad():
                if model_name == "clip":
                    hidden = get_clip_hidden_states(backbone, images)
                else:
                    hidden = backbone(
                        pixel_values        = images,
                        output_hidden_states = True,
                    ).hidden_states

            depth_pred = decoder(hidden)
            del hidden

            loss = silog_loss(depth_pred, depth_gt)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(decoder.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)

        decoder.eval()
        all_metrics = []
        with torch.no_grad():
            for batch in val_loader:
                images   = batch["image"].to(DEVICE, non_blocking=True)
                depth_gt = batch["depth"].to(DEVICE, non_blocking=True)
                if model_name == "clip":
                    hidden = get_clip_hidden_states(backbone, images)
                else:
                    hidden   = backbone(
                        pixel_values        = images,
                        output_hidden_states = True,
                    ).hidden_states
                depth_pred = decoder(hidden)
                del hidden
                all_metrics.append(compute_metrics(depth_pred, depth_gt))

        abs_rel = float(np.nanmean([m["abs_rel"] for m in all_metrics]))
        rmse    = float(np.nanmean([m["rmse"]    for m in all_metrics]))
        delta1  = float(np.nanmean([m["delta1"]  for m in all_metrics]))

        history.append({
            "epoch": epoch + 1, "train_loss": train_loss,
            "abs_rel": abs_rel, "rmse": rmse, "delta1": delta1,
        })

        print(f"  Epoch {epoch+1:3d} | loss={train_loss:.4f}  "
              f"AbsRel={abs_rel:.4f}  RMSE={rmse:.4f}  δ<1.25={delta1:.4f}")

        if abs_rel < best_abs_rel:
            best_abs_rel = abs_rel
            best_state   = {k: v.cpu().clone()
                            for k, v in decoder.state_dict().items()}

    decoder.load_state_dict(best_state)
    torch.save(best_state,
               f"/kaggle/working/phase2_{model_name}_cond{condition}.pt")

    del backbone, decoder
    torch.cuda.empty_cache()
    gc.collect()

    return history, best_abs_rel

In [9]:
TARGET_MODELS = ["dinov2", "clip", "depth_anything", "vit_small"]

In [10]:
def save_model_results(model_name):
    from IPython.display import FileLink, display
    
    results = {}
    for cond in ["A", "B", "C"]:
        path = f"/kaggle/working/phase2_{model_name}_{cond}.pt"
        results[cond] = torch.load(path)
    
    combined_path = f"/kaggle/working/phase2_{model_name}_all.pt"
    torch.save(results, combined_path)
    print(f"\nSaved all results for {model_name}")
    print(f"  A: {results['A']['best_abs_rel']:.4f}")
    print(f"  B: {results['B']['best_abs_rel']:.4f}")
    print(f"  C: {results['C']['best_abs_rel']:.4f}")
    display(FileLink(f"phase2_{model_name}_all.pt"))

In [11]:
models     = ["dinov2", "clip", "depth_anything", "vit_small"]
conditions = ["A", "B", "C"]
for m in models:
    for c in conditions:
        path = f"/kaggle/working/phase2_{m}_{c}.pt"
        exists = os.path.exists(path)
        size   = os.path.getsize(path) if exists else 0
        print(f"{m}_{c}: {'SAVED' if exists else 'MISSING'} ({size/1024:.1f} KB)")

dinov2_A: MISSING (0.0 KB)
dinov2_B: MISSING (0.0 KB)
dinov2_C: MISSING (0.0 KB)
clip_A: MISSING (0.0 KB)
clip_B: MISSING (0.0 KB)
clip_C: MISSING (0.0 KB)
depth_anything_A: MISSING (0.0 KB)
depth_anything_B: MISSING (0.0 KB)
depth_anything_C: MISSING (0.0 KB)
vit_small_A: MISSING (0.0 KB)
vit_small_B: MISSING (0.0 KB)
vit_small_C: MISSING (0.0 KB)


In [12]:
# completed = {
#     "dinov2_A":         "/kaggle/input/datasets/rcharan23/all-model-bestt/phase2_dinov2_A.pt",
#     "dinov2_B":         "/kaggle/input/datasets/rcharan23/all-model-bestt/phase2_dinov2_B.pt",
#     "dinov2_C":         "/kaggle/input/datasets/rcharan23/all-model-bestt/phase2_dinov2_C.pt",
#     "depth_anything_A": "/kaggle/input/datasets/rcharan23/all-model-bestt/phase2_depth_anything_A.pt",
#     "depth_anything_B": "/kaggle/input/datasets/rcharan23/all-model-bestt/phase2_depth_anything_B.pt",
#     "depth_anything_C": "/kaggle/input/datasets/rcharan23/all-model-bestt/phase2_depth_anything_C.pt",
#     "vit_small_A":      "/kaggle/input/datasets/rcharan23/all-model-bestt/phase2_vit_small_A.pt",
#     "vit_small_B":      "/kaggle/input/datasets/rcharan23/all-model-bestt/phase2_vit_small_B.pt",
#     "vit_small_C":      "/kaggle/input/datasets/rcharan23/all-model-bestt/phase2_vit_small_C.pt",
#     "clip_A": "/kaggle/input/datasets/rcharan23/all-model-bestt/phase2_clip_A.pt",
#     "clip_B": "/kaggle/input/datasets/rcharan23/all-model-bestt/phase2_clip_B.pt",
#     "clip_C": "/kaggle/input/datasets/rcharan23/all-model-bestt/phase2_clip_C.pt"
# }

# print("Restoring and verifying saved results:")
# for name, src in completed.items():
#     dst = f"/kaggle/working/phase2_{name}.pt"
#     if os.path.exists(src):
#         shutil.copy(src, dst)
#         data   = torch.load(dst)
#         best_absrel = data["best_abs_rel"]
#         history = data["history"]

#         # find epoch where abs_rel is closest to best_absrel
#         best_idx = min(
#             range(len(history)),
#             key=lambda i: abs(history[i]["abs_rel"] - best_absrel)
#         )

#         rmse = history[best_idx]["rmse"]
#         epochs = len(history)

#         print(f"  {name:20s} | RESTORED | AbsRel={best_absrel:.4f} | RMSE={rmse:.4f}")
#     else:
#         print(f"  {name:20s} | NOT FOUND at {src}")

Restoring and verifying saved results:
  dinov2_A             | RESTORED | AbsRel=0.1639 | RMSE=0.6512
  dinov2_B             | RESTORED | AbsRel=0.1528 | RMSE=0.6093
  dinov2_C             | RESTORED | AbsRel=0.1513 | RMSE=0.6060
  depth_anything_A     | RESTORED | AbsRel=0.1624 | RMSE=0.6336
  depth_anything_B     | RESTORED | AbsRel=0.1562 | RMSE=0.6213
  depth_anything_C     | RESTORED | AbsRel=0.1540 | RMSE=0.6162
  vit_small_A          | RESTORED | AbsRel=0.2573 | RMSE=0.9887
  vit_small_B          | RESTORED | AbsRel=0.2268 | RMSE=0.8850
  vit_small_C          | RESTORED | AbsRel=0.2147 | RMSE=0.8266
  clip_A               | RESTORED | AbsRel=0.2121 | RMSE=0.7843
  clip_B               | RESTORED | AbsRel=0.1914 | RMSE=0.7609
  clip_C               | RESTORED | AbsRel=0.1797 | RMSE=0.6988


In [ ]:
history, best = run_experiment("dinov2", "A", epochs=EPOCHS)
torch.save({"history": history, "best_abs_rel": best}, "/kaggle/working/phase2_dinov2_A.pt")
print(f"dinov2 A done | best AbsRel = {best:.4f}")

In [ ]:
history, best = run_experiment("dinov2", "B", epochs=EPOCHS)
torch.save({"history": history, "best_abs_rel": best}, "/kaggle/working/phase2_dinov2_B.pt")
print(f"dinov2 B done | best AbsRel = {best:.4f}")

In [ ]:
history, best = run_experiment("dinov2", "C", epochs=EPOCHS)
torch.save({"history": history, "best_abs_rel": best}, "/kaggle/working/phase2_dinov2_C.pt")
print(f"dinov2 C done | best AbsRel = {best:.4f}")

In [ ]:
save_model_results("dinov2")

In [ ]:
history, best = run_experiment("depth_anything", "A", epochs=EPOCHS)
torch.save({"history": history, "best_abs_rel": best}, "/kaggle/working/phase2_depth_anything_A.pt")
print(f"depth_anything A done | best AbsRel = {best:.4f}")

In [ ]:
history, best = run_experiment("depth_anything", "B", epochs=EPOCHS)
torch.save({"history": history, "best_abs_rel": best}, "/kaggle/working/phase2_depth_anything_B.pt")
print(f"depth_anything B done | best AbsRel = {best:.4f}")

In [ ]:
history, best = run_experiment("depth_anything", "C", epochs=EPOCHS)
torch.save({"history": history, "best_abs_rel": best}, "/kaggle/working/phase2_depth_anything_C.pt")
print(f"depth_anything C done | best AbsRel = {best:.4f}")

In [ ]:
save_model_results("depth_anything")

In [ ]:
history, best = run_experiment("vit_small", "A", epochs=EPOCHS)
torch.save({"history": history, "best_abs_rel": best}, "/kaggle/working/phase2_vit_small_A.pt")
print(f"vit_small A done | best AbsRel = {best:.4f}")

In [ ]:
history, best = run_experiment("vit_small", "B", epochs=EPOCHS)
torch.save({"history": history, "best_abs_rel": best}, "/kaggle/working/phase2_vit_small_B.pt")
print(f"vit_small B done | best AbsRel = {best:.4f}")

In [ ]:
history, best = run_experiment("vit_small", "C", epochs=EPOCHS)
torch.save({"history": history, "best_abs_rel": best}, "/kaggle/working/phase2_vit_small_C.pt")
print(f"vit_small C done | best AbsRel = {best:.4f}")

In [ ]:
save_model_results("vit_small")

In [ ]:
history, best = run_experiment("clip", "A", epochs=EPOCHS)
torch.save({"history": history, "best_abs_rel": best}, "/kaggle/working/phase2_clip_A.pt")
print(f"clip A done | best AbsRel = {best:.4f}")

In [ ]:
history, best = run_experiment("clip", "B", epochs=EPOCHS)
torch.save({"history": history, "best_abs_rel": best}, "/kaggle/working/phase2_clip_B.pt")
print(f"clip B done | best AbsRel = {best:.4f}")

In [ ]:
history, best = run_experiment("clip", "C", epochs=EPOCHS)
torch.save({"history": history, "best_abs_rel": best}, "/kaggle/working/phase2_clip_C.pt")
print(f"clip C done | best AbsRel = {best:.4f}")

In [ ]:
save_model_results("clip")

In [13]:
models     = ["dinov2", "clip", "depth_anything", "vit_small"]
conditions = ["A", "B", "C"]

print(f"{'Model':20s} | {'A: Final':>10} | {'B: DPT':>10} | {'C: Ours':>10} | {'B→C Gain':>10}")

for m in models:
    row = {}
    for cond in conditions:
        path = f"/kaggle/working/phase2_{m}_{cond}.pt"
        row[cond] = torch.load(path)["best_abs_rel"]
    gain = (row["B"] - row["C"]) / row["B"] * 100
    print(f"{m:20s} | {row['A']:>10.4f} | {row['B']:>10.4f} | {row['C']:>10.4f} | {gain:>9.2f}%")

Model                |   A: Final |     B: DPT |    C: Ours |   B→C Gain
dinov2               |     0.1639 |     0.1528 |     0.1513 |      0.97%
clip                 |     0.2121 |     0.1914 |     0.1797 |      6.11%
depth_anything       |     0.1624 |     0.1562 |     0.1540 |      1.43%
vit_small            |     0.2573 |     0.2268 |     0.2147 |      5.36%
